# 🔬 HAB Ablation Study
## Multi-Window LSTM Stacking Ensemble — Eastern Visayas, Philippines
### Eight ablations across four design axes: base learners · meta-learner · features · province scope

| ID | Group | What is varied |
|----|-------|----------------|
| A1 | Base learner — windows | Single-window (7d / 15d / 30d alone) vs full 3-window ensemble |
| A2 | Base learner — XGBoost | Remove XGBoost; LSTM×3 + LR only |
| A3 | Base learner — stacking gain | Best single base learner vs full ensemble |
| B1 | Meta-learner architecture | LR vs naive average vs MLP vs Random Forest |
| B2 | Threshold strategy | Fixed 0.5 · Youden J · F1-optimized · F2-optimized |
| C1 | Features — cross-province | Local-only (10 feats) vs full 26-feature set |
| C2 | Features — HAB lags | No lag features (22 feats) vs full 26-feature set |
| D1 | Province scope | Pooled regional model vs province-specific ensembles |

> **All ablations share the same base data, scaling, and splits as the main notebook.**  
> Results are collected into a single summary table at the end.


In [2]:
#%pip install tensorflow xgboost

## 0. Imports & Shared Setup

In [3]:
import os, warnings, itertools
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing  import MinMaxScaler
from sklearn.linear_model   import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.metrics        import (
    fbeta_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow.keras.models    import Model
from tensorflow.keras.layers    import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"XGBoost    : {xgb.__version__}")


TensorFlow : 2.21.0
XGBoost    : 3.2.0


## 1. Data Loading & Province Configuration

Identical to the main notebook — same CSV, same feature sets, same scalers.

In [8]:
CSV_PATH = "easternVisayas_redtide_COMPLETE.csv"
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"Rows x Cols : {df.shape}")
print(f"Date range  : {df['date'].min().date()} → {df['date'].max().date()}")

PROVINCES = {
    "EasternSamar": {"key": "eastern_samar",  "target": "toxic_red_tide_eastern_samar",  "color": "#e05252"},
    "WesternSamar": {"key": "western_samar",   "target": "toxic_red_tide_western_samar",  "color": "#5278e0"},
    "Leyte":         {"key": "leyte",            "target": "toxic_red_tide_leyte",          "color": "#52a852"},
    "Biliran":       {"key": "biliran",          "target": "toxic_red_tide_biliran",        "color": "#e09a52"},
}
ALL_KEYS = [v["key"] for v in PROVINCES.values()]
OCN_VARS = ["sea_surface_temp_C","salinity_PSU","chlorophyll_a_mg_m3","dissolved_oxygen_mmol_m3"]
CYCLIC_FEATURES = ["month_sin","month_cos","day_of_week_sin","day_of_week_cos",
                   "day_of_year_sin","day_of_year_cos"]

def province_features(key):
    ocn  = ([f"{v}_{key}" for v in OCN_VARS] +
            [f"{v}_{k}"   for v in OCN_VARS for k in ALL_KEYS if k != key])
    lags = ([f"toxic_red_tide_{key}_7d_sum"] +
            [f"toxic_red_tide_{k}_7d_sum" for k in ALL_KEYS if k != key])
    return ocn + CYCLIC_FEATURES + lags

def local_features(key):
    """10-feature local-only set (C1 ablation)."""
    ocn  = [f"{v}_{key}" for v in OCN_VARS]
    lags = [f"toxic_red_tide_{key}_7d_sum"]
    return ocn + CYCLIC_FEATURES + lags

def no_lag_features(key):
    """22-feature set without any HAB lags (C2 ablation)."""
    ocn = ([f"{v}_{key}" for v in OCN_VARS] +
           [f"{v}_{k}"   for v in OCN_VARS for k in ALL_KEYS if k != key])
    return ocn + CYCLIC_FEATURES

for name, cfg in PROVINCES.items():
    f26 = province_features(cfg["key"])
    f10 = local_features(cfg["key"])
    f22 = no_lag_features(cfg["key"])
    print(f"{name}: full={len(f26)} | local-only={len(f10)} | no-lag={len(f22)}")


Rows x Cols : (9631, 43)
Date range  : 2000-01-01 → 2026-05-14
EasternSamar: full=26 | local-only=11 | no-lag=22
WesternSamar: full=26 | local-only=11 | no-lag=22
Leyte: full=26 | local-only=11 | no-lag=22
Biliran: full=26 | local-only=11 | no-lag=22


## 2. Temporal Split, Scaling & Sequence Builder

Identical 70/15/15 split; per-province MinMaxScaler fitted on training only.

In [9]:
N         = len(df)
TRAIN_END = int(N * 0.70)
VAL_END   = int(N * 0.85)
WINDOWS   = {"7d": 7, "15d": 15, "30d": 30}
MAX_W     = 30

print(f"TRAIN : 0 → {TRAIN_END}  ({df['date'].iloc[0].date()} → {df['date'].iloc[TRAIN_END-1].date()})")
print(f"VAL   : {TRAIN_END} → {VAL_END}")
print(f"TEST  : {VAL_END} → {N}  (→ {df['date'].iloc[-1].date()})")

def build_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i+window])
        ys.append(y[i+window])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

def make_splits(X_scaled, y):
    seqs = {}
    for wn, w in WINDOWS.items():
        Xs, ys = build_sequences(X_scaled, y, w)
        diff   = MAX_W - w
        seqs[wn] = (Xs[diff:], ys[diff:])
    y_ref  = seqs["30d"][1]
    N_al   = len(y_ref)
    tr_end = TRAIN_END - MAX_W
    va_end = VAL_END   - MAX_W
    def sp(arr): return arr[:tr_end], arr[tr_end:va_end], arr[va_end:]
    lstm_splits = {}
    for wn in WINDOWS:
        Xa, ya = seqs[wn]
        Xtr,Xva,Xte = sp(Xa); ytr,yva,yte = sp(ya)
        lstm_splits[wn] = (Xtr, Xva, Xte, ytr, yva, yte)
    X_30   = seqs["30d"][0]
    X_flat = X_30.reshape(len(X_30), -1)
    Xf_tr, Xf_va, Xf_te = sp(X_flat)
    y_tr,  y_va,  y_te  = sp(y_ref)
    return dict(lstm_splits=lstm_splits, seqs_aligned=seqs,
                X_flat=X_flat, Xf_tr=Xf_tr, Xf_va=Xf_va, Xf_te=Xf_te,
                y_tr=y_tr, y_va=y_va, y_te=y_te, y_ref=y_ref,
                N_al=N_al, tr_end=tr_end, va_end=va_end)

def prepare_province(name, cfg, feat_fn=province_features):
    feats    = feat_fn(cfg["key"])
    X_raw    = df[feats].values.astype(np.float32)
    y        = df[cfg["target"]].values.astype(np.float32)
    scaler   = MinMaxScaler()
    X_sc     = X_raw.copy()
    X_sc[:TRAIN_END] = scaler.fit_transform(X_raw[:TRAIN_END])
    X_sc[TRAIN_END:] = scaler.transform(X_raw[TRAIN_END:])
    splits   = make_splits(X_sc, y)
    return dict(X_scaled=X_sc, y=y, features=feats,
                scaler=scaler, splits=splits, **cfg)

# Build main 26-feature province data (shared with all ablations that don't change features)
province_data = {n: prepare_province(n, cfg) for n, cfg in PROVINCES.items()}
print("Province data ready.")
for name, pd_ in province_data.items():
    sp = pd_["splits"]
    print(f"  {name}: train={len(sp['y_tr'])} val={len(sp['y_va'])} test={len(sp['y_te'])}")


TRAIN : 0 → 6741  (2000-01-01 → 2018-06-15)
VAL   : 6741 → 8186
TEST  : 8186 → 9631  (→ 2026-05-14)
Province data ready.
  EasternSamar: train=6711 val=1445 test=1445
  WesternSamar: train=6711 val=1445 test=1445
  Leyte: train=6711 val=1445 test=1445
  Biliran: train=6711 val=1445 test=1445


## 3. Shared Model Utilities

In [10]:
EPOCHS     = 50
BATCH_SIZE = 256

def build_lstm(window, n_features, name="lstm"):
    inp = Input(shape=(window, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = BatchNormalization()(x)
    x   = Dropout(0.3)(x)
    x   = LSTM(32)(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.3)(x)
    x   = Dense(16, activation="relu")(x)
    out = Dense(1, activation="sigmoid")(x)
    m   = Model(inputs=inp, outputs=out, name=name)
    m.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

ES   = lambda: EarlyStopping(patience=8, restore_best_weights=True, verbose=0)
RLRP = lambda: ReduceLROnPlateau(factor=0.5, patience=4, verbose=0)

def train_lstm(window, n_features, Xtr, ytr, Xva, yva, model_name="lstm"):
    m = build_lstm(window, n_features, model_name)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva),
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=[ES(), RLRP()], verbose=0)
    return m, h.history

def train_xgb(Xf_tr, y_tr, Xf_va, y_va):
    pw = float((y_tr == 0).sum()) / float(max((y_tr == 1).sum(), 1))
    m  = xgb.XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pw, eval_metric="logloss",
            early_stopping_rounds=20, random_state=SEED, verbosity=0)
    m.fit(Xf_tr, y_tr, eval_set=[(Xf_va, y_va)], verbose=False)
    return m, pw

def opt_threshold(y_true, y_prob, beta=2):
    """Return threshold that maximises F-beta on the provided labels/probs."""    
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    fb = ((1+beta**2)*prec*rec) / np.maximum(beta**2*prec + rec, 1e-9)
    best = int(np.argmax(fb))
    return float(thr[best]) if best < len(thr) else 0.5

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thr[np.argmax(j)])

def eval_at_threshold(y_true, y_prob, thresh):
    y_pred = (y_prob >= thresh).astype(int)
    return {
        "threshold" : round(thresh, 4),
        "F2"        : round(fbeta_score(y_true, y_pred, beta=2, zero_division=0), 4),
        "AUC_ROC"   : round(roc_auc_score(y_true, y_prob), 4),
        "PR_AUC"    : round(average_precision_score(y_true, y_prob), 4),
        "Recall"    : round(float(classification_report(y_true, y_pred,
                            target_names=["No HAB","HAB"],
                            output_dict=True)["HAB"]["recall"]), 4),
        "Precision" : round(float(classification_report(y_true, y_pred,
                            target_names=["No HAB","HAB"],
                            output_dict=True)["HAB"]["precision"]), 4),
        "Accuracy"  : round(float((y_true == y_pred).mean()), 4),
    }

# Collector for all results
ABLATION_RESULTS = {}
print("Utilities ready.")


Utilities ready.


---
## A0. Baseline — Full 4-Province Ensemble (Reference)

Train the full model once. Results stored in .  
All other ablations delta against this.


In [11]:
def train_full_province(name, pdata):
    sp = pdata["splits"]
    nf = len(pdata["features"])
    lstm_oof, lstm_te = {}, {}
    histories = {}
    lstm_models = {}
    for wn, w in WINDOWS.items():
        Xtr,Xva,Xte,ytr,yva,yte = sp["lstm_splits"][wn]
        m, hist = train_lstm(w, nf, Xtr, ytr, Xva, yva, f"{name}_{wn}")
        lstm_oof[wn] = m.predict(Xva, verbose=0).flatten()
        lstm_te[wn]  = m.predict(Xte, verbose=0).flatten()
        histories[wn] = hist
        lstm_models[wn] = m
        auc = roc_auc_score(sp["y_va"], lstm_oof[wn])
        ep  = int(np.argmin(hist["val_loss"])) + 1
        print(f"  LSTM-{wn}  ep={ep}  val_AUC={auc:.4f}")
    xgb_m, pw = train_xgb(sp["Xf_tr"], sp["y_tr"], sp["Xf_va"], sp["y_va"])
    xgb_oof = xgb_m.predict_proba(sp["Xf_va"])[:,1]
    xgb_te  = xgb_m.predict_proba(sp["Xf_te"])[:,1]
    print(f"  XGBoost  iter={xgb_m.best_iteration}  scale_w={pw:.2f}  val_AUC={roc_auc_score(sp['y_va'], xgb_oof):.4f}")
    meta_tr = np.column_stack([lstm_oof[k] for k in WINDOWS] + [xgb_oof])
    meta_te = np.column_stack([lstm_te[k]  for k in WINDOWS] + [xgb_te])
    meta    = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
    meta.fit(meta_tr, sp["y_va"])
    y_prob  = meta.predict_proba(meta_te)[:,1]
    return dict(lstm_models=lstm_models, lstm_oof=lstm_oof, lstm_te=lstm_te,
                xgb_model=xgb_m, xgb_oof=xgb_oof, xgb_te=xgb_te,
                meta_learner=meta, y_prob_te=y_prob, histories=histories)

print("Training Baseline (Full Ensemble)…")
province_models = {}
baseline_metrics = {}
for name, pdata in province_data.items():
    print(f"{'═'*50}{name}{'═'*50}")
    res = train_full_province(name, pdata)
    province_models[name] = res
    sp  = pdata["splits"]
    thr = opt_threshold(sp["y_te"], res["y_prob_te"], beta=2)
    baseline_metrics[name] = eval_at_threshold(sp["y_te"], res["y_prob_te"], thr)
    baseline_metrics[name]["Variant"] = "Baseline (Full)"
    baseline_metrics[name]["Province"] = name

ABLATION_RESULTS["Baseline"] = baseline_metrics
print("Baseline complete.")
display(pd.DataFrame(baseline_metrics).T[["Province","Variant","threshold","F2","AUC_ROC","PR_AUC","Recall","Precision"]])


Training Baseline (Full Ensemble)…
══════════════════════════════════════════════════EasternSamar══════════════════════════════════════════════════


ValueError: 'Eastern Samar_7d_lstm_2_lstm_cell_kernel_momentum' is not a valid scope name. A scope name has to match the following pattern: ^[A-Za-z0-9_.\\/>-]*$

---
## A1. Window Ablation — Single-Window vs Multi-Window

For each of the three windows (7d, 15d, 30d) we train a reduced ensemble:  
single LSTM + XGBoost + LR meta-learner (2 inputs instead of 4).  
The multi-window baseline is already in .

**Hypothesis:** Each window captures a distinct temporal scale. Removing any one should degrade performance, and the degradation pattern should differ by province.


In [ ]:
print("A1 — Window Ablation")
a1_results = {}

for single_win, single_w in WINDOWS.items():
    print(f"
  Window: {single_win} only")
    win_metrics = {}
    for name, pdata in province_data.items():
        sp = pdata["splits"]
        nf = len(pdata["features"])
        Xtr,Xva,Xte,ytr,yva,yte = sp["lstm_splits"][single_win]
        m, _ = train_lstm(single_w, nf, Xtr, ytr, Xva, yva, f"{name}_abl_{single_win}")
        p_oof = m.predict(Xva, verbose=0).flatten()
        p_te  = m.predict(Xte, verbose=0).flatten()
        xgb_m, _ = train_xgb(sp["Xf_tr"], sp["y_tr"], sp["Xf_va"], sp["y_va"])
        xgb_oof  = xgb_m.predict_proba(sp["Xf_va"])[:,1]
        xgb_te   = xgb_m.predict_proba(sp["Xf_te"])[:,1]
        # 2-input meta (single LSTM + XGBoost)
        meta_tr  = np.column_stack([p_oof, xgb_oof])
        meta_te  = np.column_stack([p_te,  xgb_te])
        meta     = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
        meta.fit(meta_tr, sp["y_va"])
        y_prob   = meta.predict_proba(meta_te)[:,1]
        thr      = opt_threshold(sp["y_te"], y_prob, beta=2)
        m_out    = eval_at_threshold(sp["y_te"], y_prob, thr)
        m_out["Variant"]  = f"A1: LSTM-{single_win} only"
        m_out["Province"] = name
        win_metrics[name] = m_out
        print(f"    {name:15s}  F2={m_out['F2']:.4f}  AUC={m_out['AUC_ROC']:.4f}  PR={m_out['PR_AUC']:.4f}")
    a1_results[f"A1_{single_win}"] = win_metrics

ABLATION_RESULTS.update(a1_results)
print("
A1 complete.")


### A1 — Results Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("A1: Window Ablation — F2-Score vs Province", fontweight="bold", fontsize=13)

metrics_to_compare = ["F2", "AUC_ROC", "PR_AUC"]
titles = ["F2-Score (primary)", "AUC-ROC", "PR-AUC"]

for ax, metric, title in zip(axes, metrics_to_compare, titles):
    x     = np.arange(len(PROVINCES))
    width = 0.2
    # Baseline
    bl_vals = [ABLATION_RESULTS["Baseline"][n][metric] for n in PROVINCES]
    ax.bar(x - 1.5*width, bl_vals, width, label="Baseline (3-window)", color="#333", alpha=0.85)
    colors_w = ["#e05252", "#5278e0", "#52a852"]
    for i, wn in enumerate(WINDOWS):
        vals = [ABLATION_RESULTS[f"A1_{wn}"][n][metric] for n in PROVINCES]
        ax.bar(x + (i - 0.5)*width, vals, width, label=f"LSTM-{wn} only",
               color=colors_w[i], alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(list(PROVINCES.keys()), fontsize=8)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0.7, 1.02)
    ax.legend(fontsize=7)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("ablation_A1_window.png", dpi=120, bbox_inches="tight")
plt.show()


---
## A2. XGBoost Removal — LSTM×3 + LR Only

Drop XGBoost entirely. The meta-learner receives only .

**Hypothesis:** XGBoost captures feature interaction patterns across the 780-dimensional flat window that the sequential LSTMs under-weight. Removing it should degrade most where XGBoost's individual val-AUC was furthest above the best LSTM.


In [ ]:
print("A2 — XGBoost Removal")
a2_results = {}

for name, pdata in province_data.items():
    sp = pdata["splits"]
    nf = len(pdata["features"])
    lstm_oof, lstm_te = {}, {}
    for wn, w in WINDOWS.items():
        Xtr,Xva,Xte,ytr,yva,yte = sp["lstm_splits"][wn]
        m, _ = train_lstm(w, nf, Xtr, ytr, Xva, yva, f"{name}_a2_{wn}")
        lstm_oof[wn] = m.predict(Xva, verbose=0).flatten()
        lstm_te[wn]  = m.predict(Xte, verbose=0).flatten()
    # 3-input meta (LSTMs only, no XGBoost)
    meta_tr = np.column_stack([lstm_oof[k] for k in WINDOWS])
    meta_te = np.column_stack([lstm_te[k]  for k in WINDOWS])
    meta    = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
    meta.fit(meta_tr, sp["y_va"])
    y_prob  = meta.predict_proba(meta_te)[:,1]
    thr     = opt_threshold(sp["y_te"], y_prob, beta=2)
    m_out   = eval_at_threshold(sp["y_te"], y_prob, thr)
    m_out["Variant"]  = "A2: LSTM×3 only (no XGB)"
    m_out["Province"] = name
    a2_results[name] = m_out
    print(f"  {name:15s}  F2={m_out['F2']:.4f}  AUC={m_out['AUC_ROC']:.4f}  PR={m_out['PR_AUC']:.4f}")

ABLATION_RESULTS["A2_no_xgb"] = a2_results
print("A2 complete.")


---
## A3. Stacking Gain — Best Single Base Learner vs Full Ensemble

No retraining required — uses existing test-set probabilities from the Baseline.  
Each base learner is evaluated independently at its own F2-optimized threshold.

**Hypothesis:** The stacking ensemble should outperform the best individual base learner on F2-Score and PR-AUC, validating the architectural complexity.


In [ ]:
print("A3 — Stacking Gain (zero retraining)")
a3_results = {}

for name, pdata in province_data.items():
    sp   = pdata["splits"]
    base = province_models[name]
    y_te = sp["y_te"]
    row  = {}
    # Individual LSTM base learners
    for wn in WINDOWS:
        prob = base["lstm_te"][wn]
        thr  = opt_threshold(y_te, prob, beta=2)
        m    = eval_at_threshold(y_te, prob, thr)
        m["Variant"]  = f"A3: LSTM-{wn} solo"
        m["Province"] = name
        row[f"LSTM_{wn}"] = m
    # XGBoost solo
    prob = base["xgb_te"]
    thr  = opt_threshold(y_te, prob, beta=2)
    m    = eval_at_threshold(y_te, prob, thr)
    m["Variant"]  = "A3: XGBoost solo"
    m["Province"] = name
    row["XGBoost"] = m
    a3_results[name] = row
    best_solo = max(row.values(), key=lambda x: x["F2"])
    ens_f2    = ABLATION_RESULTS["Baseline"][name]["F2"]
    print(f"  {name}: best_solo={best_solo['Variant']} F2={best_solo['F2']:.4f} | Ensemble F2={ens_f2:.4f}  gain={ens_f2-best_solo['F2']:+.4f}")

ABLATION_RESULTS["A3_solo"] = a3_results
print("A3 complete.")


### A3 — Stacking Gain Plot

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("A3: Stacking Gain — F2-Score per Base Learner vs Ensemble",
             fontweight="bold", fontsize=13)

solo_keys   = list(WINDOWS.keys()) + ["XGBoost"]
solo_labels = [f"LSTM-{w}" for w in WINDOWS] + ["XGBoost"]

for ax, (name, cfg) in zip(axes, PROVINCES.items()):
    solo_f2  = [a3_results[name][k]["F2"] for k in [f"LSTM_{w}" for w in WINDOWS] + ["XGBoost"]]
    ens_f2   = ABLATION_RESULTS["Baseline"][name]["F2"]
    bar_cols = ["#aec6e8"]*3 + ["#c8a2c8"]
    bars = ax.bar(solo_labels, solo_f2, color=bar_cols, edgecolor="white")
    ax.axhline(ens_f2, color=cfg["color"], lw=2.5, ls="--", label=f"Ensemble ({ens_f2:.4f})")
    ax.set_title(name, fontweight="bold")
    ax.set_ylim(max(0, min(solo_f2)-0.05), 1.02)
    ax.set_ylabel("F2-Score")
    ax.tick_params(axis="x", labelsize=8)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, solo_f2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f"{val:.3f}", ha="center", fontsize=7)

plt.tight_layout()
plt.savefig("ablation_A3_stacking_gain.png", dpi=120, bbox_inches="tight")
plt.show()


---
## B1. Meta-Learner Architecture — LR vs Naive Average vs MLP vs Random Forest

The 4-dimensional OOF probability vector is fixed (from Baseline).  
Only the meta-learner is swapped. No base learner retraining.

| Variant | Meta-learner |
|---------|-------------|
| B1a | Naive average (no learning) |
| B1b | Logistic Regression (baseline) |
| B1c | MLP (hidden=8, 1 layer) |
| B1d | Random Forest (n=100) |


In [ ]:
print("B1 — Meta-Learner Architecture")

META_VARIANTS = {
    "B1a_avg"  : "Naive average",
    "B1b_LR"   : "Logistic Regression",
    "B1c_MLP"  : "MLP (8 hidden)",
    "B1d_RF"   : "Random Forest (100)",
}
b1_results = {k: {} for k in META_VARIANTS}

for name, pdata in province_data.items():
    sp   = pdata["splits"]
    base = province_models[name]
    y_va = sp["y_va"]
    y_te = sp["y_te"]

    # OOF and test stacks (already trained in baseline)
    oof_stack = np.column_stack([base["lstm_oof"][k] for k in WINDOWS] + [base["xgb_oof"]])
    te_stack  = np.column_stack([base["lstm_te"][k]  for k in WINDOWS] + [base["xgb_te"]])

    # B1a — naive average
    y_prob = te_stack.mean(axis=1)
    thr    = opt_threshold(y_te, y_prob, beta=2)
    m      = eval_at_threshold(y_te, y_prob, thr)
    m["Variant"] = "B1a: Naive average"; m["Province"] = name
    b1_results["B1a_avg"][name] = m

    # B1b — LR (already baseline, recompute cleanly)
    lr = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
    lr.fit(oof_stack, y_va)
    y_prob = lr.predict_proba(te_stack)[:,1]
    thr    = opt_threshold(y_te, y_prob, beta=2)
    m      = eval_at_threshold(y_te, y_prob, thr)
    m["Variant"] = "B1b: Logistic Regression"; m["Province"] = name
    b1_results["B1b_LR"][name] = m

    # B1c — MLP
    mlp = MLPClassifier(hidden_layer_sizes=(8,), max_iter=500, random_state=SEED)
    mlp.fit(oof_stack, y_va)
    y_prob = mlp.predict_proba(te_stack)[:,1]
    thr    = opt_threshold(y_te, y_prob, beta=2)
    m      = eval_at_threshold(y_te, y_prob, thr)
    m["Variant"] = "B1c: MLP (hidden=8)"; m["Province"] = name
    b1_results["B1c_MLP"][name] = m

    # B1d — Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight="balanced")
    rf.fit(oof_stack, y_va)
    y_prob = rf.predict_proba(te_stack)[:,1]
    thr    = opt_threshold(y_te, y_prob, beta=2)
    m      = eval_at_threshold(y_te, y_prob, thr)
    m["Variant"] = "B1d: Random Forest"; m["Province"] = name
    b1_results["B1d_RF"][name] = m

    print(f"  {name}: avg={b1_results['B1a_avg'][name]['F2']:.4f} | "
          f"LR={b1_results['B1b_LR'][name]['F2']:.4f} | "
          f"MLP={b1_results['B1c_MLP'][name]['F2']:.4f} | "
          f"RF={b1_results['B1d_RF'][name]['F2']:.4f}")

ABLATION_RESULTS.update(b1_results)
print("B1 complete.")


---
## B2. Threshold Strategy — Fixed 0.5 · Youden J · F1-optimized · F2-optimized

Zero retraining. Uses Baseline ensemble probabilities.  
Directly quantifies the public-health cost of suboptimal threshold selection.

**Hypothesis:** F2-optimization should maximize recall (PSP priority). Fixed 0.5 will suppress recall especially on Eastern Samar (optimal threshold 0.157) and Biliran.


In [ ]:
print("B2 — Threshold Strategy")

THR_VARIANTS = {
    "B2a_fixed05"  : "Fixed 0.5",
    "B2b_youden"   : "Youden J",
    "B2c_F1"       : "F1-optimized",
    "B2d_F2"       : "F2-optimized (baseline)",
}
b2_results = {k: {} for k in THR_VARIANTS}

print(f"{'Province':<16} {'Fixed-0.5':>10} {'Youden-J':>10} {'F1-opt':>10} {'F2-opt':>10}  [F2-Score]")
print("─" * 65)

for name, pdata in province_data.items():
    y_te   = pdata["splits"]["y_te"]
    y_prob = province_models[name]["y_prob_te"]

    thresholds = {
        "B2a_fixed05" : 0.5,
        "B2b_youden"  : youden_threshold(y_te, y_prob),
        "B2c_F1"      : opt_threshold(y_te, y_prob, beta=1),
        "B2d_F2"      : opt_threshold(y_te, y_prob, beta=2),
    }
    f2s = []
    for key, thr in thresholds.items():
        m = eval_at_threshold(y_te, y_prob, thr)
        m["Variant"]  = THR_VARIANTS[key]
        m["Province"] = name
        b2_results[key][name] = m
        f2s.append(m["F2"])
    print(f"  {name:<14}  {f2s[0]:>10.4f} {f2s[1]:>10.4f} {f2s[2]:>10.4f} {f2s[3]:>10.4f}")

ABLATION_RESULTS.update(b2_results)

# Recall comparison plot — most impactful for reviewers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("B2: Threshold Strategy — Recall & F2-Score Comparison", fontweight="bold")

for ax, metric in zip(axes, ["Recall", "F2"]):
    x     = np.arange(len(PROVINCES))
    width = 0.18
    thr_colors = ["#d9534f", "#f0ad4e", "#5bc0de", "#5cb85c"]
    for i, (key, label) in enumerate(THR_VARIANTS.items()):
        vals = [b2_results[key][n][metric] for n in PROVINCES]
        ax.bar(x + (i-1.5)*width, vals, width, label=label, color=thr_colors[i], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(list(PROVINCES.keys()), fontsize=8)
    ax.set_title(metric, fontweight="bold")
    ax.set_ylim(0.5, 1.05)
    ax.legend(fontsize=7)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("ablation_B2_threshold.png", dpi=120, bbox_inches="tight")
plt.show()
print("B2 complete.")


---
## C1. Cross-Province Features — Local-Only (10 feats) vs Full (26 feats)

Retrain full ensemble per province using only local OCN + cyclical + local lag (10 features).  
Removes the 12 cross-province OCN features and 3 cross-province HAB lags.

**Hypothesis:** Biliran should degrade most — with only 622 positive training days, it likely relies on Eastern Samar's bloom signal as a leading spatial indicator.


In [ ]:
print("C1 — Cross-Province Feature Removal")

# Build 10-feature province data
pdata_local = {n: prepare_province(n, cfg, feat_fn=local_features)
               for n, cfg in PROVINCES.items()}

c1_results = {}
for name, pdata in pdata_local.items():
    sp = pdata["splits"]
    nf = len(pdata["features"])
    print(f"
  {name}  ({nf} features)")
    lstm_oof, lstm_te = {}, {}
    for wn, w in WINDOWS.items():
        Xtr,Xva,Xte,ytr,yva,yte = sp["lstm_splits"][wn]
        m, _ = train_lstm(w, nf, Xtr, ytr, Xva, yva, f"{name}_c1_{wn}")
        lstm_oof[wn] = m.predict(Xva, verbose=0).flatten()
        lstm_te[wn]  = m.predict(Xte, verbose=0).flatten()
    xgb_m, pw = train_xgb(sp["Xf_tr"], sp["y_tr"], sp["Xf_va"], sp["y_va"])
    xgb_oof   = xgb_m.predict_proba(sp["Xf_va"])[:,1]
    xgb_te    = xgb_m.predict_proba(sp["Xf_te"])[:,1]
    meta_tr   = np.column_stack([lstm_oof[k] for k in WINDOWS] + [xgb_oof])
    meta_te   = np.column_stack([lstm_te[k]  for k in WINDOWS] + [xgb_te])
    meta      = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
    meta.fit(meta_tr, sp["y_va"])
    y_prob    = meta.predict_proba(meta_te)[:,1]
    thr       = opt_threshold(sp["y_te"], y_prob, beta=2)
    m_out     = eval_at_threshold(sp["y_te"], y_prob, thr)
    m_out["Variant"]  = "C1: Local-only (10 feats)"
    m_out["Province"] = name
    c1_results[name] = m_out
    delta_f2  = m_out["F2"] - ABLATION_RESULTS["Baseline"][name]["F2"]
    print(f"    F2={m_out['F2']:.4f}  AUC={m_out['AUC_ROC']:.4f}  PR={m_out['PR_AUC']:.4f}  ΔF2={delta_f2:+.4f}")

ABLATION_RESULTS["C1_local"] = c1_results
print("C1 complete.")


---
## C2. HAB Lag Feature Removal — No Lag Features (22 feats) vs Full (26 feats)

Removes all four 7-day rolling HAB sums (local + 3 cross-province).  
Tests whether the model is learning true oceanographic bloom dynamics  
or primarily exploiting temporal autocorrelation as a persistence shortcut.

**Hypothesis:** If performance drops sharply, the model is largely a bloom-persistence predictor.  
If it holds up, the oceanographic features carry the primary predictive signal.


In [ ]:
print("C2 — HAB Lag Feature Removal")

pdata_nolag = {n: prepare_province(n, cfg, feat_fn=no_lag_features)
               for n, cfg in PROVINCES.items()}

c2_results = {}
for name, pdata in pdata_nolag.items():
    sp = pdata["splits"]
    nf = len(pdata["features"])
    print(f"
  {name}  ({nf} features)")
    lstm_oof, lstm_te = {}, {}
    for wn, w in WINDOWS.items():
        Xtr,Xva,Xte,ytr,yva,yte = sp["lstm_splits"][wn]
        m, _ = train_lstm(w, nf, Xtr, ytr, Xva, yva, f"{name}_c2_{wn}")
        lstm_oof[wn] = m.predict(Xva, verbose=0).flatten()
        lstm_te[wn]  = m.predict(Xte, verbose=0).flatten()
    xgb_m, pw = train_xgb(sp["Xf_tr"], sp["y_tr"], sp["Xf_va"], sp["y_va"])
    xgb_oof   = xgb_m.predict_proba(sp["Xf_va"])[:,1]
    xgb_te    = xgb_m.predict_proba(sp["Xf_te"])[:,1]
    meta_tr   = np.column_stack([lstm_oof[k] for k in WINDOWS] + [xgb_oof])
    meta_te   = np.column_stack([lstm_te[k]  for k in WINDOWS] + [xgb_te])
    meta      = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
    meta.fit(meta_tr, sp["y_va"])
    y_prob    = meta.predict_proba(meta_te)[:,1]
    thr       = opt_threshold(sp["y_te"], y_prob, beta=2)
    m_out     = eval_at_threshold(sp["y_te"], y_prob, thr)
    m_out["Variant"]  = "C2: No lag features (22 feats)"
    m_out["Province"] = name
    c2_results[name] = m_out
    delta_f2  = m_out["F2"] - ABLATION_RESULTS["Baseline"][name]["F2"]
    print(f"    F2={m_out['F2']:.4f}  AUC={m_out['AUC_ROC']:.4f}  PR={m_out['PR_AUC']:.4f}  ΔF2={delta_f2:+.4f}")

ABLATION_RESULTS["C2_nolag"] = c2_results
print("C2 complete.")


---
## D1. Province-Specific vs Pooled Regional Model

Train a **single** ensemble on all four provinces combined.  
A one-hot province indicator (4 columns) is appended as additional features.  
Evaluate per-province test-set performance and compare against province-specific models.

**Hypothesis:** Pooled training dilutes Biliran's minority bloom signal across the combined dataset  
where positive prevalence drops to ~15% overall. Province-specific ensembles should win on F2 and PR-AUC.


In [ ]:
print("D1 — Pooled Regional vs Province-Specific")

# Build pooled dataset with province one-hot encoding
prov_list = list(PROVINCES.keys())
n_prov    = len(prov_list)

all_X_parts, all_y_parts, all_prov_idx = [], [], []
for pidx, (name, cfg) in enumerate(PROVINCES.items()):
    feats  = province_features(cfg["key"])
    X_raw  = df[feats].values.astype(np.float32)
    y      = df[cfg["target"]].values.astype(np.float32)
    onehot = np.zeros((len(X_raw), n_prov), dtype=np.float32)
    onehot[:, pidx] = 1.0
    all_X_parts.append(np.hstack([X_raw, onehot]))
    all_y_parts.append(y)
    all_prov_idx.append(np.full(len(y), pidx, dtype=int))

# Stack vertically and re-sort by date (interleaved across provinces)
X_pool_raw  = np.vstack(all_X_parts)
y_pool      = np.concatenate(all_y_parts)
prov_idx_all= np.concatenate(all_prov_idx)

# Scale pooled data — fit only on training rows
scaler_pool = MinMaxScaler()
X_pool = X_pool_raw.copy()
X_pool[:TRAIN_END*n_prov] = scaler_pool.fit_transform(X_pool_raw[:TRAIN_END*n_prov])
X_pool[TRAIN_END*n_prov:] = scaler_pool.transform(X_pool_raw[TRAIN_END*n_prov:])

# Build sequences for pooled data
NF_POOL = X_pool.shape[1]  # 26 + 4 = 30
pool_splits = make_splits(X_pool[:N], y_pool[:N])   # use aligned single-province length for sequences
# Note: for the pooled model we use the first province's length as reference;
# evaluation is done per-province using province index masks applied to test predictions.

# Train pooled ensemble
print(f"  Pooled feature dim: {NF_POOL}")
print(f"  Pooled dataset size: {N} rows (shared date axis, province indicated by one-hot)")

# We retrain one ensemble on the pooled data
pool_lstm_oof, pool_lstm_te = {}, {}
for wn, w in WINDOWS.items():
    Xtr,Xva,Xte,ytr,yva,yte = pool_splits["lstm_splits"][wn]
    m, _ = train_lstm(w, NF_POOL, Xtr, ytr, Xva, yva, f"pool_{wn}")
    pool_lstm_oof[wn] = m.predict(Xva, verbose=0).flatten()
    pool_lstm_te[wn]  = m.predict(Xte, verbose=0).flatten()
    print(f"  Pool LSTM-{wn}  val_AUC={roc_auc_score(yva, pool_lstm_oof[wn]):.4f}")

pool_xgb, _ = train_xgb(pool_splits["Xf_tr"], pool_splits["y_tr"],
                          pool_splits["Xf_va"], pool_splits["y_va"])
pool_xgb_oof = pool_xgb.predict_proba(pool_splits["Xf_va"])[:,1]
pool_xgb_te  = pool_xgb.predict_proba(pool_splits["Xf_te"])[:,1]

pool_meta_tr = np.column_stack([pool_lstm_oof[k] for k in WINDOWS] + [pool_xgb_oof])
pool_meta_te = np.column_stack([pool_lstm_te[k]  for k in WINDOWS] + [pool_xgb_te])
pool_meta    = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
pool_meta.fit(pool_meta_tr, pool_splits["y_va"])
pool_y_prob  = pool_meta.predict_proba(pool_meta_te)[:,1]

# Evaluate per-province using the one-hot column in Xte to identify province rows
d1_results = {}
# Rebuild per-province test masks from the pooled aligned test set
Xte_pool    = pool_splits["Xf_te"]   # (N_te, 30*NF_POOL)
y_te_pool   = pool_splits["y_te"]

# The one-hot columns are at positions [-4:] of each lag step; we use last step
# Simpler: re-build province membership from the original date-index positions
# The pooled X was built by stacking provinces sequentially (not interleaved by date).
# So test rows 0..N_te correspond to province 0, N_te..2*N_te to province 1, etc.
N_te_single = len(province_data["Eastern Samar"]["splits"]["y_te"])

for pidx, name in enumerate(PROVINCES.keys()):
    start = pidx * N_te_single
    end   = start + N_te_single
    y_te_p = y_te_pool[start:end]
    y_pr_p = pool_y_prob[start:end]
    thr    = opt_threshold(y_te_p, y_pr_p, beta=2)
    m_out  = eval_at_threshold(y_te_p, y_pr_p, thr)
    m_out["Variant"]  = "D1: Pooled regional"
    m_out["Province"] = name
    d1_results[name] = m_out
    spec_f2 = ABLATION_RESULTS["Baseline"][name]["F2"]
    print(f"  {name:15s}  Pooled F2={m_out['F2']:.4f}  SpecificF2={spec_f2:.4f}  Δ={m_out['F2']-spec_f2:+.4f}")

ABLATION_RESULTS["D1_pooled"] = d1_results
print("D1 complete.")


---
## Summary — Full Ablation Results Table

Consolidated delta table showing change relative to the Baseline for F2, AUC-ROC, PR-AUC, and Recall.  
Green = improvement over baseline · Red = degradation.


In [ ]:
# Flatten all results into a tidy dataframe
rows = []

def add_rows(key, variant_label, result_dict, is_nested=False):
    if is_nested:
        # a3 structure: {province: {subkey: metrics}}
        for prov, sub in result_dict.items():
            for subkey, m in sub.items():
                r = dict(m)
                r["Ablation"] = key
                r["Variant"]  = r.get("Variant", subkey)
                r["Province"] = prov
                rows.append(r)
    else:
        for prov, m in result_dict.items():
            r = dict(m)
            r["Ablation"] = key
            rows.append(r)

add_rows("Baseline",   "Baseline (Full)",   ABLATION_RESULTS["Baseline"])
add_rows("A1_7d",      "A1: LSTM-7d only",  ABLATION_RESULTS["A1_7d"])
add_rows("A1_15d",     "A1: LSTM-15d only", ABLATION_RESULTS["A1_15d"])
add_rows("A1_30d",     "A1: LSTM-30d only", ABLATION_RESULTS["A1_30d"])
add_rows("A2_no_xgb",  "A2: No XGBoost",    ABLATION_RESULTS["A2_no_xgb"])
add_rows("A3_solo",    "A3: Solo learners",  ABLATION_RESULTS["A3_solo"], is_nested=True)
add_rows("B1a_avg",    "B1a: Naive avg",     ABLATION_RESULTS["B1a_avg"])
add_rows("B1b_LR",     "B1b: LR",            ABLATION_RESULTS["B1b_LR"])
add_rows("B1c_MLP",    "B1c: MLP",           ABLATION_RESULTS["B1c_MLP"])
add_rows("B1d_RF",     "B1d: RF",            ABLATION_RESULTS["B1d_RF"])
add_rows("B2a_fixed05","B2a: Fixed 0.5",     ABLATION_RESULTS["B2a_fixed05"])
add_rows("B2b_youden", "B2b: Youden J",      ABLATION_RESULTS["B2b_youden"])
add_rows("B2c_F1",     "B2c: F1-opt",        ABLATION_RESULTS["B2c_F1"])
add_rows("B2d_F2",     "B2d: F2-opt",        ABLATION_RESULTS["B2d_F2"])
add_rows("C1_local",   "C1: Local-only",     ABLATION_RESULTS["C1_local"])
add_rows("C2_nolag",   "C2: No lag feats",   ABLATION_RESULTS["C2_nolag"])
add_rows("D1_pooled",  "D1: Pooled",         ABLATION_RESULTS["D1_pooled"])

summary_df = pd.DataFrame(rows)
cols_show  = ["Ablation","Province","Variant","threshold","F2","AUC_ROC","PR_AUC","Recall","Precision","Accuracy"]
summary_df = summary_df[[c for c in cols_show if c in summary_df.columns]]
summary_df.to_csv("ablation_summary.csv", index=False)

# Delta table vs baseline
bl_df   = summary_df[summary_df["Ablation"]=="Baseline"][["Province","F2","AUC_ROC","PR_AUC","Recall"]].rename(
          columns={"F2":"BL_F2","AUC_ROC":"BL_AUC","PR_AUC":"BL_PR","Recall":"BL_Rec"})
delta_df= summary_df.merge(bl_df, on="Province")
for metric, bl_col in [("F2","BL_F2"),("AUC_ROC","BL_AUC"),("PR_AUC","BL_PR"),("Recall","BL_Rec")]:
    delta_df[f"Δ{metric}"] = (delta_df[metric] - delta_df[bl_col]).round(4)
delta_df = delta_df[["Ablation","Province","Variant","F2","ΔF2","AUC_ROC","ΔAUC_ROC","PR_AUC","ΔPR_AUC","Recall","ΔRecall"]]
delta_df.to_csv("ablation_delta_summary.csv", index=False)

print("Full summary:")
display(summary_df.set_index(["Ablation","Province"]))


### Delta Heatmap — F2-Score Change vs Baseline

In [ ]:
# Pivot: rows = variant, cols = province
pivot = delta_df.pivot_table(index="Variant", columns="Province", values="ΔF2", aggfunc="mean")
# Re-order columns
prov_order = list(PROVINCES.keys())
pivot = pivot[[p for p in prov_order if p in pivot.columns]]

fig, ax = plt.subplots(figsize=(10, max(8, len(pivot)*0.42)))
sns.heatmap(
    pivot, annot=True, fmt=".4f", center=0,
    cmap="RdYlGn", linewidths=0.5, linecolor="white",
    ax=ax, cbar_kws={"label": "ΔF2 vs Baseline"}
)
ax.set_title("Ablation ΔF2-Score vs Baseline (green = improvement, red = degradation)",
             fontweight="bold", fontsize=12)
ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(fontsize=10); plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.savefig("ablation_heatmap_deltaF2.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: ablation_heatmap_deltaF2.png")
print("Saved: ablation_summary.csv")
print("Saved: ablation_delta_summary.csv")


---
## Interpretation Guide

| Ablation | Key question answered |
|----------|-----------------------|
| **A1** | Which temporal window(s) carry the most predictive signal per province? |
| **A2** | Does XGBoost add value beyond the three LSTMs? |
| **A3** | How much does stacking gain over the best individual model? |
| **B1** | Is logistic regression the right meta-learner, or does a nonlinear combiner help? |
| **B2** | What is the public-health cost of suboptimal threshold selection? |
| **C1** | Do cross-province features contribute beyond local oceanographic context? |
| **C2** | Is the model learning bloom dynamics or bloom persistence? |
| **D1** | Is province-specific training necessary, or does a pooled model suffice? |

> **For the paper:** cite ΔF2 and ΔRecall columns from  directly in the ablation subsection table.  
> The heatmap is Fig. 13. A3 and B2 can be presented without retraining caveats since they require no new model fitting.
